# Introduction

These jupyter notebooks will act as an introduction on how to use root, this one will provide basic training on creating and filling histograms as well as exploring root files and extracting data for analysis.

The notebooks work using individual cells such as this which contain either markdown text or root code. Each cell can be ran individually to execute their contents. Run each cell one after another to ensure the guides in this notebook work as expected.

Try running the cell below, this allows the root histogram outputs to be interactive which will be used later.

In [1]:
%jsroot on

In [2]:
gSystem->pwd()

(const char *) "/home/userfs/e/ee772/Work/York_Nuclear_Techniques_2026"


The cell above should print the directory in which this notebook currently thinks it's running in.

# ROOT

## What is ROOT?

ROOT is a common software framework used within high energy physics with building blocks for data processing, analysis, visualisation and storage.

Root is mainly written in C++ but has support of python scripts via relavant modules.

Root contains its own built-in interpreter : CLING, this provides the software with an interactive C++ shell. The user can load macros for multi line commands or the root command line can be used with one line commands such as using it as a calculator.

All code written in the code cells are interpreted by root, try using the cell below to use root as a calculator:

In [3]:
18 + 24

(int) 42


Commands can either be written in single line or multiline forms, e.g. : 

In [4]:
for (int i=3; i>0; i--) {
    cout << i << endl;
};
cout << "BOOM!!" << endl;

3
2
1
BOOM!!


Note that ROOT does not care about line indentation to execute commands (but you should try to tab code appropriately for readability).

Each line must end with a semi-colon - `;`

## Macros

Writing multiline commands can be dificult especially if an error is made in a previous line and the whole command needs to be restarted. 

Instead we can write macros using a text editor to carry out a task. These are .C files that contain syntax to be interpreted by the software.

The cell below loads the file `demoMacro.C` with an argument of 10. 

The macro doubles the input value then outputs the result to the root terminal.

In [15]:
.L demoMacro.C

The next cell after loading the macro executes it. To execute a macro it first needs to be loaded. From here an argument can be passed to it.

In [18]:
demoMacro(55.5,45.)

(double) 121.50000


As well as running the macro, open it and take a look at how the code looks inside.

### Task - Modifying our demoMacro

Edit the macro so it does the following:

1. Rather than multiplying the input by 2, multiply it by 3.
2. Add a second argument to the macro, subtract this argument from the result of the initial multiplication.

### Compiling Macros

Another method of executing a macro it to use the `.x demoMacro.C(NUMBER)` command. Instead of compiling the macro this method interprets it on the fly. Try using this in the cell below and pass an argument of 100 to the macro.

The choice to Compile vs Interpret depends on the context of the situation. Compiling the macro will mean faster execution. As a rule of thumb;
- Lots of code or running slow? Compile!
- Does it load C++ standard library? Compile!
- Does it behave weird? Compile!
- Is there an error you cannot find? Compile !

## Drawing Functions

Root contains multiple classes which allow for different things to be acomplished. Using the `TF1` class can allow for 1D functions to be drawn. The cell below will draw the function sin(x) from 0 to 10.

In [27]:
TCanvas * c1 = new TCanvas("c", "c", 600, 600);
auto f = new TF1("func", "cos(x)", 0, 2*M_PI);
cout << TMath::DegToRad()*360 << endl;
f->Draw();
c1->Draw();

6.28319


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


Each class has its own use, following the naming scheme for TF1, TF2(3) can be used to draw 2 dimensional (and 3 dimensional) functions.
Note that we have used `auto` to define the object type for us here. This is *usually* ok, but if you know what object you want, it's probably better to specifically define your object as such. If you're unsure, you can specify `auto` then check the object type afterwards.

When using root in a command line interface, the canvas can be updated in real time using additional commands. When using this notebook, options can be changed but then the graph must manually be redrawn. Try using the cells below to change the colour of the line the function is plot with.

In [24]:
f->SetLineColor(4);
f->Draw();
c1->Draw();

### Task - Switching the function

Change the function we're drawing from sin(x) to cosine(x). Modify the drawing range to be from 0 to 360 degrees.

**Hint** - What is the X-axis in our original plot? You can use the `TMath::RadToDeg()` function to convert from radians to degrees - `TMath::RadToDeg()*x`.

### Aside - Choosing Plot Colours

When we just want to display a single data series or fit, the choice of colour usually doesn't matter too much. However, try to avoid bright colours on white backgrounds in these cases. Particularly if you're showing the plot as part of a presentation.

When showing multiple fits or data series though, you should take care when choosing your colours. There are two primary considerations here:

1. You should choose colours that print in greyscale in a **distinguishable manner**. This is often a **requirement** of journals for publication.
2. Take care to choose colours that are accessible to people with colour vision deficiency (CVD - aka colour blindness). In particular, **avoid red and green** as much as possible.

~10% of men and ~0.5% of women have some form of CVD. 

Luckily, a set of accessible colours is already pre-defined in root from version 6.34 onwards. There are 3 sets of 6/8/10 colours which fit both of our criteria above. 

[See this blog post](https://root.cern/blog/new-color-schemes/) for a full description of these colour sets. I would recommend just getting in the habit of using these where possible. They are very easy to use, try the cell below for example to change the colours of our function from earlier.


In [28]:
f->SetLineColor("kP6Red");
f->Draw();
c1->Draw();

In [29]:
f->SetLineColor("kP6Blue");
f->Draw();
c1->Draw();

In [30]:
f->SetLineColor("kP6Yellow");
f->Draw();
c1->Draw();

Note that if you're using an older version of ROOT (or python), you could just use the same colours by using their hexcodes:

1. kP6Blue - #5790fc
2. kP6Yellow - #f89c20
3. kP6Red - #e42536
4. kP6Grape - #964a8b
5. kP6Gray - #9c9ca1
6. kP6Violet - #7021dd

For the kP6 set for example.

In [31]:
f->SetLineColor(TColor::GetColor("#5790fc"));
f->Draw();
c1->Draw();

Note that we've had to do something a bit tedious here to get the correct colour as the argument for `SetLineColour` is expecting an object of type `Color_t`. Our hex code is **NOT** a `Color_t` so instead we have to use `GetColor` on the hex code to return a `Color_t`.

The joys of object orientated programming.

## Histograms

Histograms contain binned data, this is often the most important class for physicists within ROOT. To create a one dimensional, float precision the `TH1f` class is used.

The example below creates a one dimensional histogram called my hist with axis titles Bins and Entries. There are 10 evenly spacedd bins which range from values 0 to 20. 

The histogram is filled at values 3.5 and 7.5.

In [32]:
TCanvas * c1 = new TCanvas("c", "c", 600, 600);
auto h = new TH1F("hist", "My Hist;Bins;Entries", 10, 0, 20);
h->Fill(3.5);
h->Fill(7.5);
h->Draw();
c1->Draw();

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


The histogram can be updated on the fly via the command line to adjust parameters such as changing the range or number of bins to better suit the data that has been obtained. An example is seen below which changes the range of the x-axis:

In [33]:
h->GetXaxis()->SetRangeUser(0, 10);
h->Draw();
c1->Draw();

Some comments and things to think about:

- Looking at **just** the histogram, what can we say about the two entries we now have?
- Do we know the original values we entered from the histogram?
- Is our binning range appropriate?
- Is our bin width appropriate?
- What happens if we fill a value outside of our chosen range?

Using the previous section as a guide, load and execute the `demoHistogram.C` file in the cell below. This should create a histogram containing a gausian to be used.

In [34]:
.x demoHistogram.C

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


A fitting line can be applied to this distribution in two ways, either by using the command line with a function like `demo->Fit("gaus")` or by using the fit panel on the graph.

The fit panel in a ROOT TBrowser is accessed by right clicking on the histogram and selecting fit panel. Then select the function and click fit.

<b> NOTE: The %jsroot on command does not enable every X-Window feature available from within the ROOT command line. There’s no TBrowser, Treeviewer, or FitPanel in notebooks </b>


### Task - Fit our histogram and make some changes

Open up the demoHistogram.C macro and modify it to do the following:

1. Modify the macro to fit a gaussian to our histogram.
2. Rather than filling with a pre-defined number of events, make the number of events filled in the histogram an input argument.
3. Run the macro with a range of events and consider the impact on the fit parameters.
4. Change the bin widths and see how your fit vaires.

In [52]:
.x demoHistogram.C(15000)

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      99.4609
NDf                       =          101
Edm                       =  2.11347e-06
NCalls                    =           53
Constant                  =      429.653   +/-   4.31453     
Mean                      =   0.00256034   +/-   0.0568834   
Sigma                     =      6.92149   +/-   0.0404425    	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Warning in <TROOT::Append>: Replacing existing TH1: h (Potential memory leak).


In [53]:
.x demoHistogram.C(10)

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      0.42886
NDf                       =            6
Edm                       =  2.02585e-07
NCalls                    =          108
Constant                  =      1.14738   +/-   0.567365    
Mean                      =     0.547637   +/-   9.21313     
Sigma                     =      10.7125   +/-   24.505       	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Warning in <TROOT::Append>: Replacing existing TH1: h (Potential memory leak).


In [49]:
.x demoHistogram.C(100);

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       21.514
NDf                       =           39
Edm                       =  6.12573e-08
NCalls                    =          102
Constant                  =      2.35643   +/-   0.377762    
Mean                      =      1.11533   +/-   2.06806     
Sigma                     =      11.2985   +/-   3.26717      	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Warning in <TROOT::Append>: Replacing existing TH1: h (Potential memory leak).


In [50]:
.x demoHistogram.C(1000)

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       94.356
NDf                       =           76
Edm                       =   7.8292e-06
NCalls                    =           61
Constant                  =      26.1048   +/-   1.13968     
Mean                      =   -0.0999211   +/-   0.238711    
Sigma                     =      6.97276   +/-   0.209185     	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Warning in <TROOT::Append>: Replacing existing TH1: h (Potential memory leak).


In [51]:
.x demoHistogram.C(1000000)

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      117.392
NDf                       =          127
Edm                       =  4.02828e-09
NCalls                    =           52
Constant                  =      28471.8   +/-   34.886      
Mean                      = -0.000318687   +/-   0.00700609  
Sigma                     =      7.00513   +/-   0.00495985   	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Warning in <TROOT::Append>: Replacing existing TH1: h (Potential memory leak).


## Files and the TBrowser

### Files

Using the class TFile, this allows us to store any ROOT object to the disk. Using the previously created histogram `h`, this can be stored by using the following cells.

In [54]:
auto file = TFile::Open("file.root", "RECREATE");
h->Write();
file->Close();

If you look in your directory now, you should now have a file called "file.root". It will contain our histogram from earlier.

The main options when opening a file are as follows:
- NEW or CREATE - Creates a new file and opens for writing, if the file already exists then the file is not opened
- RECREATE - Creates a new file, if the file already exists then it is overwritten
- UPDATE - Opens an existing file for writing. If no file exists then one is created
- READ - Open an existing file for reading

### TBrowser

The TBrowser has many uses such as opening files, navigating within them and searching TTrees. To open the TBrowser the syntax `new TBrowser` is used as seen in the cell below

**Opening TBrowser does not work within the notebooks directly. For any steps including them please go to File->New->Terminal**

OR

Open a terminal and navigate to this directory there.

**From here use the command `root` within the terminal to start root. From here use the command in the cell below to access the TBrowser.**
    

In [ ]:
new TBrowser

Navigate to the created file called `file.root` From here you should be able to open the previous histogram.

### Task - Using the TBrowser

Take a minute or two to see what you can do in the TBrowser. Some ideas:

- Try changing the colour of your histogram and fit directly from the TBrowser.
- Set the x and y scales to be logarithmic.
- Change the axes titles.
- Change the plot title.
- Rebin the histogram.

**Thought** - Can we increase the number of bins our histogram has? Why or why not?

## ROOT Trees

Often times the data being proccessed in root is very large as it is the results from multiple detectors reading many variables for each particle. This means that we need something to store this data in an efficient way.

Root Trees have been designed to support very large collections of objects.

The Trees allow for direct and random access to an entry (where sequential access will be the most efficient).

The class TTree is the main container for data storage. It can store any class and basic type. Trees are structured into branches and leaves. Using a TBrowser you can read a subset of all branches

- To browse a TTree use the TBrowser.
- To analyse a TTree use the TTreeViewer.

There is an example of a Tree called `Day_1_Tutorial_Input.root` that you can explore using the TBrowser. 

This file is in the Nuclear Techniques drive that was shared with you. Download it and move it to the directory you're working in.

Once downloaded, try having a search through now. Within the folder `events;2`, filters can be applied using wildcard expressions. The symbol for this is in the top left of the browser.

This is the file that will be used within the **next** tutorial so familiarise yourself with it and explore the different branches it contains and how the tree is structured.

We will explore how events are stored in trees, how we can access them and how we can apply cuts to them.

# Summary Task

Starting from scratch or from the previous macro:

- Create and fit three **separate** 1D histograms. Fill the histograms with N random entries, sampled from a gaussian distribution with a $\mu$ of -1 and $\sigma$ of 5. The three histograms in question should use N = 100, 1000 and 10000.
- Save the three histograms to a ROOT file.
- Modify the script to provide the filename via an input argument.
    - **Hint** - What *type* of object should the input file be? A `string`? `Char`?
- Create a canvas showing your highest event sample distribution with all three fits overlaid simultaneously. Scale the 100 and 1000 event fits up to 10000.
    - **Hint** - You can provide arguments to the `Draw` command. `Draw("SAME")` can be used to draw on an existing canvas.
    - **Hint** - The constant term of a Gaussian determines its scale. In the predefined Gaussian, this is Parameter 0. Check [this tutorial](https://root.cern.ch/root/htmldoc/guides/users-guide/FittingHistograms.html) for more on fit options.
- Save your canvas as a `.pdf` type file. Name this pdf based upon your input argument.
  - **Hint** - Use the `Print()` function.

Feel free to choose and adjust the binning of your histogram as you see fit. **Consider how this impacts your fit.**

Extension tasks:

- Create a legend for your plot, clearly labelling all three fits.
  - **Hint** - Look up the `TLegend` class.
- Draw your histogram with error bars.
- Add a small loop which checks if your file exists before trying to write to it.
    - **Hint** - `gSystem->AccessPathName(FILEPATH)` checks if a file in `FILEPATH` exits or not. If it does, it returns `FALSE`. Can you use this to construct a routine that checks if your file is present?

# Some Thoughts/Help

As with most coding tasks, I strongly recommend learning through doing. **Try** things, see what works and what doesn't. Many problems in ROOT boil down to a few common causes:

- Trying to provide the wrong **type** of object to a function.
  - Use the [ROOT class reference](https://root.cern.ch/doc/v640/index.html) to see which arguments a function actually expects.
  - Try looking up a TH1 - Look at the valid ways you could construct a TH1. Equally, how can you fill it?
- Referencing or filling an object that doesn't exist. This usually results in a `Segmentation Fault` - aka a seg fault
    - As an example, if you have defined a histogram called `myHisto1` but then try to fill or draw `myHisto2` and it doesn't exist, **your macro will almost certainly crash**
    - Check carefully for copy paste errors.
- Another common error is array indices being out of bounds. If you're looping over 10k events for example and filling a histogram with these events. Your code will not be happy if you ask for event 10012. **It doesn't exist!**
- Avoid hard coding files where possible. For example, if our macro always writes to `file.root` - this could be a recipe for disaster.
    - You may overwrite a file created from a different macro or get weird behaviour.
    - Good practice would be to have a filename constructed from input arguments.
    - Even better is to check if a file exists **before** doing anything else in your macro. If it does, prompt the user to check if they want to overwrite what they have!

We'll explore more of these next time, there are some links to further resources below too.

## Additional Resources

ROOT is a fairly mature way to analyse NP and HEP data. There are various resources available. A non-exhaustive list includes:

- [Snippets Directory](https://github.com/sjdkay/York_Nuclear_Techniques/tree/main/Snippets)
    - I created a set of minimal macro examples for a few tasks you might want to carry out in ROOT. Feel free to copy and modify these as you like.
- [HSF Training Centre](https://hsf-training.org/training-center/)
  - The HEP Software Foundation/IRIS-HEP has a range of tutorials and resources available, for ROOT and for other common scientific coding languages.
    - Be aware that many of these make use of PyROOT, so some modification may be needed.
    - The examples here also typically focus on using ROOT/Python in a very modern (I.e. Dataframe orientated) way
- [The ROOT Reference](https://root.cern/doc/v640/)
    - Straight from the horse's mouth! The class refernece should be your go-to source for figuring out what arguments a given class can handle and what you can do with a given class etc
    - This is usually the first result if you google `CERN ROOT TH1D` for example.
        - When searching, I'd recommend always prefixing with `CERN ROOT` for better results.
- [CERN ROOT Tutorials](https://root.cern/doc/v640/group__Tutorials.html)
  - ROOT has a large library of tutorials for you to browse and try too.